# 07 LPIPS Metrics

This notebook computes perceptual similarity metrics for the OpenCV Telea restoration outputs.

It produces:

- `outputs/metrics/metrics_opencv_telea_lpips.csv`
- `outputs/metrics/metrics_opencv_telea_with_lpips.csv`

Run this notebook after:

1. dataset verification
2. preprocessing
3. mask generation
4. OpenCV restoration
5. classical metrics

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Source directory: D:\Masters\FH\Thesis\painting-restoration-eval\src


## 1. Imports and paths

In [12]:
import pandas as pd

from restoration_eval.paths import (
    CLEAN_DIR,
    MASK_DIR,
    RESTORED_OPENCV_DIR,
    PROCESSED_METADATA_DIR,
    METRICS_DIR,
    ensure_directories,
)

from restoration_eval.metrics_lpips import (
    get_device,
    load_lpips_model,
    compute_lpips_for_restorations,
    merge_lpips_with_classical_metrics,
    summarize_lpips_by_mask_type,
    rank_worst_lpips_cases,
)

ensure_directories()
METRICS_DIR.mkdir(parents=True, exist_ok=True)

RESTORATION_METADATA_PATH = PROCESSED_METADATA_DIR / "metadata_restorations_opencv_telea.csv"
CLASSICAL_METRICS_PATH = METRICS_DIR / "metrics_opencv_telea_classical.csv"

print("Restoration metadata:", RESTORATION_METADATA_PATH)
print("Classical metrics:", CLASSICAL_METRICS_PATH)

Restoration metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restorations_opencv_telea.csv
Classical metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\metrics_opencv_telea_classical.csv


## 2. Load metadata and classical metrics

In [13]:
restoration_metadata = pd.read_csv(RESTORATION_METADATA_PATH)
classical_metrics = pd.read_csv(CLASSICAL_METRICS_PATH)

restoration_metadata.columns = restoration_metadata.columns.str.strip()
classical_metrics.columns = classical_metrics.columns.str.strip()

display(restoration_metadata)
display(classical_metrics)

print("Restoration rows:", len(restoration_metadata))
print("Classical metric rows:", len(classical_metrics))

,painting_id,mask_type,model_name,inpaint_radius,clean_filename,mask_filename,masked_filename,restored_filename,mask_area_ratio
0,p001,irregular_small,opencv_telea,3,p001_clean.png,p001_irregular_small_mask.png,p001_irregular_small_masked.png,p001_irregular_small_restored_opencv_telea.png,0.0254
1,p001,scratch_lines,opencv_telea,3,p001_clean.png,p001_scratch_lines_mask.png,p001_scratch_lines_masked.png,p001_scratch_lines_restored_opencv_telea.png,0.0233
2,p001,irregular_large,opencv_telea,3,p001_clean.png,p001_irregular_large_mask.png,p001_irregular_large_masked.png,p001_irregular_large_restored_opencv_telea.png,0.1134
3,p002,irregular_small,opencv_telea,3,p002_clean.png,p002_irregular_small_mask.png,p002_irregular_small_masked.png,p002_irregular_small_restored_opencv_telea.png,0.0274
4,p002,scratch_lines,opencv_telea,3,p002_clean.png,p002_scratch_lines_mask.png,p002_scratch_lines_masked.png,p002_scratch_lines_restored_opencv_telea.png,0.0243
5,p002,irregular_large,opencv_telea,3,p002_clean.png,p002_irregular_large_mask.png,p002_irregular_large_masked.png,p002_irregular_large_restored_opencv_telea.png,0.1129
6,p003,irregular_small,opencv_telea,3,p003_clean.png,p003_irregular_small_mask.png,p003_irregular_small_masked.png,p003_irregular_small_restored_opencv_telea.png,0.0254
7,p003,scratch_lines,opencv_telea,3,p003_clean.png,p003_scratch_lines_mask.png,p003_scratch_lines_masked.png,p003_scratch_lines_restored_opencv_telea.png,0.0158
8,p003,irregular_large,opencv_telea,3,p003_clean.png,p003_irregular_large_mask.png,p003_irregular_large_masked.png,p003_irregular_large_restored_opencv_telea.png,0.1112


,painting_id,mask_type,model_name,mask_area_ratio,mae,mse,psnr,ssim,mask_mae,mask_mse,mask_psnr
0,p001,irregular_small,opencv_telea,0.0254,0.119746,0.987967,48.183379,0.994996,4.714594,38.897717,32.231564
1,p001,scratch_lines,opencv_telea,0.0233,0.112400,1.660638,45.928055,0.994513,4.823983,71.271484,29.601646
2,p001,irregular_large,opencv_telea,0.1134,1.414233,36.205166,32.543098,0.967264,12.476045,319.393890,23.087538
3,p002,irregular_small,opencv_telea,0.0274,0.197944,2.610738,43.963171,0.992953,7.235071,95.425148,28.334177
4,p002,scratch_lines,opencv_telea,0.0243,0.168409,8.616002,38.777745,0.994116,6.940446,355.081390,22.627525
5,p002,irregular_large,opencv_telea,0.1129,0.999594,15.664153,36.181734,0.969453,8.857144,138.795975,26.707035
6,p003,irregular_small,opencv_telea,0.0254,0.375600,10.905023,37.754538,0.986486,14.768215,428.774353,21.808517
7,p003,scratch_lines,opencv_telea,0.0158,0.102845,1.957686,45.213373,0.995667,6.505827,123.840660,27.202169
8,p003,irregular_large,opencv_telea,0.1112,2.292114,84.357819,28.869550,0.937255,20.611731,758.583740,19.330769


Restoration rows: 9
Classical metric rows: 9


## 3. Load LPIPS model

In [14]:
device = get_device(prefer_cuda=True)
print("Using device:", device)

lpips_model = load_lpips_model(net="alex", device=device)

Using device: cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\lpips\lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(mode

## 4. Compute LPIPS metrics

In [15]:
lpips_df = compute_lpips_for_restorations(
    restoration_metadata=restoration_metadata,
    clean_dir=CLEAN_DIR,
    mask_dir=MASK_DIR,
    restored_dir=RESTORED_OPENCV_DIR,
    lpips_model=lpips_model,
    device=device,
    crop_padding=48,
    crop_resize=256,
)

display(lpips_df)

print("LPIPS rows:", len(lpips_df))

,painting_id,mask_type,model_name,lpips_full,lpips_mask_crop,crop_left,crop_upper,crop_right,crop_lower
0,p001,irregular_small,opencv_telea,0.022993,0.213292,390,111,649,370
1,p001,scratch_lines,opencv_telea,0.010144,0.002203,0,0,697,697
2,p001,irregular_large,opencv_telea,0.091059,0.237288,295,75,733,513
3,p002,irregular_small,opencv_telea,0.022857,0.207872,91,179,357,445
4,p002,scratch_lines,opencv_telea,0.011867,0.006967,0,62,706,768
5,p002,irregular_large,opencv_telea,0.085263,0.249581,1,90,442,531
6,p003,irregular_small,opencv_telea,0.021101,0.205288,443,433,694,684
7,p003,scratch_lines,opencv_telea,0.004369,0.001808,0,0,768,768
8,p003,irregular_large,opencv_telea,0.091533,0.331011,190,211,612,633


LPIPS rows: 9


## 5. Save LPIPS metrics

In [16]:
LPIPS_METRICS_PATH = METRICS_DIR / "metrics_opencv_telea_lpips.csv"

lpips_df.to_csv(LPIPS_METRICS_PATH, index=False)

print("Saved:", LPIPS_METRICS_PATH)

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\metrics_opencv_telea_lpips.csv


## 6. Merge LPIPS with classical metrics

In [17]:
metrics_with_lpips = merge_lpips_with_classical_metrics(
    classical_metrics=classical_metrics,
    lpips_df=lpips_df,
)

METRICS_WITH_LPIPS_PATH = METRICS_DIR / "metrics_opencv_telea_with_lpips.csv"

metrics_with_lpips.to_csv(METRICS_WITH_LPIPS_PATH, index=False)

display(metrics_with_lpips)

print("Saved:", METRICS_WITH_LPIPS_PATH)

,painting_id,mask_type,model_name,mask_area_ratio,mae,mse,psnr,ssim,mask_mae,mask_mse,mask_psnr,lpips_full,lpips_mask_crop,crop_left,crop_upper,crop_right,crop_lower
0,p001,irregular_small,opencv_telea,0.0254,0.119746,0.987967,48.183379,0.994996,4.714594,38.897717,32.231564,0.022993,0.213292,390,111,649,370
1,p001,scratch_lines,opencv_telea,0.0233,0.112400,1.660638,45.928055,0.994513,4.823983,71.271484,29.601646,0.010144,0.002203,0,0,697,697
2,p001,irregular_large,opencv_telea,0.1134,1.414233,36.205166,32.543098,0.967264,12.476045,319.393890,23.087538,0.091059,0.237288,295,75,733,513
3,p002,irregular_small,opencv_telea,0.0274,0.197944,2.610738,43.963171,0.992953,7.235071,95.425148,28.334177,0.022857,0.207872,91,179,357,445
4,p002,scratch_lines,opencv_telea,0.0243,0.168409,8.616002,38.777745,0.994116,6.940446,355.081390,22.627525,0.011867,0.006967,0,62,706,768
5,p002,irregular_large,opencv_telea,0.1129,0.999594,15.664153,36.181734,0.969453,8.857144,138.795975,26.707035,0.085263,0.249581,1,90,442,531
6,p003,irregular_small,opencv_telea,0.0254,0.375600,10.905023,37.754538,0.986486,14.768215,428.774353,21.808517,0.021101,0.205288,443,433,694,684
7,p003,scratch_lines,opencv_telea,0.0158,0.102845,1.957686,45.213373,0.995667,6.505827,123.840660,27.202169,0.004369,0.001808,0,0,768,768
8,p003,irregular_large,opencv_telea,0.1112,2.292114,84.357819,28.869550,0.937255,20.611731,758.583740,19.330769,0.091533,0.331011,190,211,612,633


Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\metrics_opencv_telea_with_lpips.csv


## 7. Summary by mask type

In [18]:
summary_lpips_by_mask = summarize_lpips_by_mask_type(metrics_with_lpips)

display(summary_lpips_by_mask)

,mask_type,cases,mean_mask_mae,mean_mask_psnr,mean_full_ssim,mean_lpips_full,mean_lpips_mask_crop
2,scratch_lines,3,6.090086,26.477114,0.994765,0.008793,0.003659
1,irregular_small,3,8.905960,27.458086,0.991479,0.022317,0.208817
0,irregular_large,3,13.981640,23.041780,0.957991,0.089285,0.272627


## 8. Worst cases by crop LPIPS

In [19]:
worst_lpips_cases = rank_worst_lpips_cases(metrics_with_lpips)

display(worst_lpips_cases)

,painting_id,mask_type,model_name,mask_mae,mask_psnr,ssim,lpips_full,lpips_mask_crop
8,p003,irregular_large,opencv_telea,20.611731,19.330769,0.937255,0.091533,0.331011
5,p002,irregular_large,opencv_telea,8.857144,26.707035,0.969453,0.085263,0.249581
2,p001,irregular_large,opencv_telea,12.476045,23.087538,0.967264,0.091059,0.237288
0,p001,irregular_small,opencv_telea,4.714594,32.231564,0.994996,0.022993,0.213292
3,p002,irregular_small,opencv_telea,7.235071,28.334177,0.992953,0.022857,0.207872
6,p003,irregular_small,opencv_telea,14.768215,21.808517,0.986486,0.021101,0.205288
4,p002,scratch_lines,opencv_telea,6.940446,22.627525,0.994116,0.011867,0.006967
1,p001,scratch_lines,opencv_telea,4.823983,29.601646,0.994513,0.010144,0.002203
7,p003,scratch_lines,opencv_telea,6.505827,27.202169,0.995667,0.004369,0.001808


## 9. Sanity checks

In [20]:
expected_rows = len(restoration_metadata)

assert len(lpips_df) == expected_rows, f"Expected {expected_rows} LPIPS rows, got {len(lpips_df)}"
assert len(metrics_with_lpips) == len(classical_metrics), "Merged metrics row count changed unexpectedly."

required_columns = [
    "lpips_full",
    "lpips_mask_crop",
    "crop_left",
    "crop_upper",
    "crop_right",
    "crop_lower",
]

missing_columns = [col for col in required_columns if col not in metrics_with_lpips.columns]
assert not missing_columns, f"Missing columns: {missing_columns}"

assert metrics_with_lpips["lpips_full"].notna().all(), "Missing full-image LPIPS values."
assert metrics_with_lpips["lpips_mask_crop"].notna().all(), "Missing crop LPIPS values."

print("Sanity checks passed.")

Sanity checks passed.
